# Evaluating Retrieval Systems demo notebook


In [2]:
!pip install rank_bm25
import json
import numpy as np
import pandas as pd
from scipy.stats import hypergeom

## Chapter 1. Success@K, Precision@K, Recall@K

Key idea is to start with one ranking and one set of relevant documents.  
At depth `K`, count how many relevant documents appear in the retrieved set.


In [4]:
ranked = ["doc_5", "doc_3", "doc_8", "doc_1", "doc_9"]
# ranked contains the retriever's ordered results
# K means we evaluate only the first K results
relevant = {"doc_2", "doc_5", "doc_8"}
K = 3

top_k = ranked[:K]

hits = 0
for doc_id in top_k:
    if doc_id in relevant:
        hits += 1

success_at_k = 1 if hits > 0 else 0
precision_at_k = hits / K
recall_at_k = hits / len(relevant)

print("Success@3 =", success_at_k)
print("Precision@3 =", precision_at_k)
print("Recall@3 =", recall_at_k)


Success@3 = 1
Precision@3 = 0.6666666666666666
Recall@3 = 0.6666666666666666


### Real example: BEIR SciFact

SciFact example: Each query is a scientific claim. The qrels identify the document(s) judged relevant to that claim. Below we show one query and its gold relevant document.

Qrel (query-relevance judgment): a label that tells us which document(s) are considered relevant for a given query, sometimes with a relevance score.

In [11]:
!wget -q -nc https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip
!unzip -qo scifact.zip

queries = pd.read_json("scifact/queries.jsonl", lines=True)
corpus = pd.read_json("scifact/corpus.jsonl", lines=True)
qrels = pd.read_csv("scifact/qrels/test.tsv", sep="\t")

# Find query 1, .iloc[0] takes the first matching row
query = queries[queries["_id"] == 1].iloc[0]
# means: filter to query 1, then take the first matching row.

# Qrel = relevance judgment: which documents are relevant for a query
gold = qrels[qrels["query-id"] == 1]

print("Query:")
print(query["text"])

# Get the IDs of the documents marked relevant for this query
gold_doc_ids = gold["corpus-id"]

# .isin(...) checks which corpus document IDs appear in gold_doc_ids
gold_docs = corpus[corpus["_id"].isin(gold_doc_ids)]
# means: for each corpus row, check whether its _id is one of the gold document IDs.

print("\nGold relevant document:")
gold_docs[["_id", "title"]]

Query:
0-dimensional biomaterials show inductive properties.

Gold relevant document:


,_id,title
4241,31715818,New opportunities: the use of nanotechnologies...


## Chapter 2. MRR, MAP, nDCG

Here, rank position matters.  

MRR: Only cares about the position of the very first correct item and ignores everything after it.
- If the first relevant result is at position 1, the score is 1.0, at position 2, it is 0.5, and so on.
---
NDCG: Evaluates the position and order of all items down to depth K, heavily penalizing good items buried lower down.
- It measures the ranking quality using a logarithmic position discount compared against an ideal ordering
---
Mean Average Precision (MAP): Measures the quality of the entire ranked list by rewarding systems that place all relevant items near the top.
- It calculates precision at every rank where a relevant item appears and averages those values for a query.



In [22]:
!wget -q -nc https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nfcorpus.zip
!unzip -qo nfcorpus.zip

nf_queries = pd.read_json("nfcorpus/queries.jsonl", lines=True)
nf_corpus = pd.read_json("nfcorpus/corpus.jsonl", lines=True)
nf_qrels = pd.read_csv("nfcorpus/qrels/test.tsv", sep="\t")

qid = "PLAIN-2"

# Find the selected query; .iloc[0] takes the first matching row
query = nf_queries[nf_queries["_id"] == qid].iloc[0]

# Get all relevance judgments for this query
# NFCorpus uses graded relevance, so score can indicate different levels of relevance
judgments = nf_qrels[nf_qrels["query-id"] == qid]

print("Query:")
print(query["text"])

print("\nRelevance judgments for this query:")
print("Higher score = more relevant")

judgments[["corpus-id", "score"]].head(10)

Query:
Do Cholesterol Statin Drugs Cause Breast Cancer?

Relevance judgments for this query:
Higher score = more relevant


,corpus-id,score
0,MED-2427,2
1,MED-10,2
2,MED-2429,2
3,MED-2430,2
4,MED-2431,2
5,MED-14,2
6,MED-2432,2
7,MED-2428,1
8,MED-2440,1
9,MED-2434,1


In [25]:
# Small example ranking for the NFCorpus query above
# Relevance: 2 = highly relevant, 1 = relevant, 0 = not relevant

ranked = ["doc_A", "doc_B", "doc_C", "doc_D"]
relevance = [2, 0, 1, 1]

print("Rank 1:", ranked[0], "| relevance =", relevance[0])
print("Rank 2:", ranked[1], "| relevance =", relevance[1])
print("Rank 3:", ranked[2], "| relevance =", relevance[2])
print("Rank 4:", ranked[3], "| relevance =", relevance[3])

Rank 1: doc_A | relevance = 2
Rank 2: doc_B | relevance = 0
Rank 3: doc_C | relevance = 1
Rank 4: doc_D | relevance = 1


In [26]:
# ranked and relevance come from the example above
# ranked    = ["doc_A", "doc_B", "doc_C", "doc_D"]
# relevance = [2,       0,       1,       1]


# --------------------------------------------------
# Reciprocal Rank (RR)
# Where does the FIRST relevant result appear?
# --------------------------------------------------

rr = 0

for rank in range(len(ranked)):
    if relevance[rank] > 0:
        rr = 1 / (rank + 1)
        break

print("Reciprocal Rank =", rr)


# --------------------------------------------------
# Average Precision (AP)
# Are the relevant results concentrated near the top?
# --------------------------------------------------

hits = 0
precision_sum = 0

for rank in range(len(ranked)):
    if relevance[rank] > 0:
        hits = hits + 1
        precision_at_this_rank = hits / (rank + 1)
        precision_sum = precision_sum + precision_at_this_rank

ap = precision_sum / hits

print("Average Precision =", ap)


# --------------------------------------------------
# nDCG
# Are the MOST relevant results near the top?
# --------------------------------------------------

dcg = 0

for rank in range(len(ranked)):
    gain = relevance[rank]
    discount = np.log2(rank + 2)
    dcg = dcg + gain / discount


# Best possible ranking: put the highest relevance grades first
ideal_relevance = sorted(relevance, reverse=True)

ideal_dcg = 0

for rank in range(len(ideal_relevance)):
    gain = ideal_relevance[rank]
    discount = np.log2(rank + 2)
    ideal_dcg = ideal_dcg + gain / discount

ndcg = dcg / ideal_dcg

print("nDCG =", ndcg)

Reciprocal Rank = 1.0
Average Precision = 0.8055555555555555
nDCG = 0.9360403422435027


## Chapter 3. Bits-over-Random

A raw retrieval score does not tell us how difficult the retrieval problem was.  
`N`, `R`, and `K` determine how likely random selection was to succeed.


In [27]:
N = 3633
R = 38
K = 10
observed_success = 0.72

p_random = hypergeom.sf(0, N, R, K)
bits_over_random = np.log2(observed_success / p_random)
ceiling = -np.log2(p_random)

print("P(random success) =", p_random)
print("Bits over random  =", bits_over_random)
print("Maximum possible  =", ceiling)


P(random success) = 0.09992643341349307
Bits over random  = 2.8490586386381644
Maximum possible  = 3.322989826970577


### Same corpus but different retrieval unit

Split the NFCorpus documents into ~300-word passages.  
The source material is the same, but `N` and the number of relevant units `R` now change.

NFCorpus normally retrieves whole documents. For this example, we create a second index by breaking each document into smaller chunks of about 300 words and retrieving those chunks instead.

The underlying content and query stay the same, but the retrieval unit changes:

- document retrieval: each whole document is one candidate
- passage retrieval: each 300-word chunk is one candidate

The same raw retrieval score can mean different things when the retrieval unit changes. Bits-over-Random accounts for the difficulty of the underlying retrieval space.


In [33]:
# --------------------------------------------------
# Same corpus, two different retrieval units
# Documents vs ~100-word passages
# --------------------------------------------------

# Create a simple passage-level version of NFCorpus.
# Each document is treated as one or more ~100-word retrieval chunks.
# 100 words is only an illustrative chunk size chosen to make
# the change in retrieval granularity easy to see.

text = nf_corpus["title"].fillna("") + " " + nf_corpus["text"].fillna("")
word_count = text.str.split().str.len()

passage_count = np.ceil(word_count / 100)
passage_count = np.maximum(1, passage_count)

nf_corpus["passages"] = passage_count.astype(int)


# Get the documents judged relevant for our example query
query_qrels = nf_qrels[nf_qrels["query-id"] == qid]
relevant_rows = query_qrels[query_qrels["score"] > 0]
relevant_doc_ids = relevant_rows["corpus-id"]


# --------------------------------------------------
# Document-level retrieval
# N = all candidate documents
# R = relevant documents
# --------------------------------------------------

N_doc = len(nf_corpus)
R_doc = len(relevant_doc_ids)


# --------------------------------------------------
# Passage-level retrieval
# N = all candidate passages
#
# For this simple example, assume each relevant document
# contains one passage with the evidence needed for the query.
#
# The number of candidates N grows because documents are split,
# but R does not automatically grow with it.
# --------------------------------------------------

N_passage = nf_corpus["passages"].sum()
R_passage = R_doc


# Compare the two retrieval spaces at the SAME K
K = 10

# Hold observed retrieval success fixed so we isolate
# the effect of changing the retrieval unit, N, and R.
observed_success = 0.70


# Document-level random baseline and Bits-over-Random
p_random_doc = hypergeom.sf(0, N_doc, R_doc, K)
bor_doc = np.log2(observed_success / p_random_doc)


# Passage-level random baseline and Bits-over-Random
p_random_passage = hypergeom.sf(0, N_passage, R_passage, K)
bor_passage = np.log2(observed_success / p_random_passage)


print("Assume both systems achieve Success@10 = 0.70")

print("\nDOCUMENT RETRIEVAL")
print("N =", N_doc, "candidate documents")
print("R =", R_doc, "relevant documents")
print("K =", K)
print("Random success =", round(p_random_doc, 4))
print("Bits-over-Random =", round(bor_doc, 2))

print("\nPASSAGE RETRIEVAL")
print("N =", N_passage, "candidate passages")
print("R =", R_passage, "relevant passages")
print("K =", K)
print("Random success =", round(p_random_passage, 4))
print("Bits-over-Random =", round(bor_passage, 2))

Assume both systems achieve Success@10 = 0.70

DOCUMENT RETRIEVAL
N = 3633 candidate documents
R = 24 relevant documents
K = 10
Random success = 0.0642
Bits-over-Random = 3.45

PASSAGE RETRIEVAL
N = 10270 candidate passages
R = 24 relevant passages
K = 10
Random success = 0.0231
Bits-over-Random = 4.92


## Chapter 4. Choosing K for agent tool retrieval

For one correct tool, `R = 1`, so the random baseline becomes simply `K / N`.  
As more tools are shown to the LLM, random success rises and the selectivity ceiling falls.


In [35]:
!wget -q -nc -O BFCL_v3_simple.json https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard/resolve/main/BFCL_v3_simple.json

# Load the BFCL examples
bfcl = []

for line in open("BFCL_v3_simple.json"):
    bfcl.append(json.loads(line))

# Count the unique tools in the registry
tool_names = set()

for row in bfcl:
    for tool in row["function"]:
        tool_names.add(tool["name"])


# Use the first example
example = bfcl[0]

query = example["question"][0][0]["content"]
gold_tool = example["function"][0]["name"]

print("Query:", query)
print("Gold tool:", gold_tool)
print("Registry size N:", len(tool_names))

Query: Find the area of a triangle with a base of 10 units and height of 5 units.
Gold tool: calculate_triangle_area
Registry size N: 370


In [37]:
# N = total number of tools in the registry
N = len(tool_names)

# Compare a few possible retrieval depths K
depths = [1, 5, 7, 20, 50, N]

rows = []

for K in depths:
    # With one correct tool, random success is simply K / N
    p_random = K / N

    # Maximum Bits-over-Random if the correct tool is retrieved
    max_bits = -np.log2(p_random)

    rows.append({
        "K": K,
        "P(random)": round(p_random, 3),
        "Max Bits-over-Random": round(max_bits, 2)
    })

# Show a clean table without the pandas row index
df = pd.DataFrame(rows)
display(df.style.hide(axis="index"))

K,P(random),Max Bits-over-Random
1,0.003000,8.530000
5,0.014000,6.210000
7,0.019000,5.720000
20,0.054000,4.210000
50,0.135000,2.890000
370,1.000000,-0.000000


### Downstream result from the tool-depth paper

| strategy | gold tool retrieved | LLM chooses gold when shown | end-to-end success | avg. K |
|---|---:|---:|---:|---:|
| BoR adaptive | 76.9% | 93.1% | 71.7% | 2.2 |
| fixed K=5 | 84.2% | 87.1% | 73.3% | 5.0 |
| fixed K=1 | 65.0% | 100.0% | 65.0% | 1.0 |